In [1]:
"""
================================================================================
PROJECT: Student Information Registry
File: Student_Registry_Manager.py
--------------------------------------------------------------------------------
Initial Implementation Section:
  - Set up the temporary student registry
  - Load saved information from JSON with exception handling
  - Display stored entries in a structured format
  - Provide the command-line menu framework

Implementation Tasks:
  1. Finish add_student_record() with validation
  2. Build search_student_record() for ID and partial-name lookup
  3. Build delete_student_record() with confirmation
  4. Add update_student_record() for modifying details
  5. Implement save_records_to_json() for persistent storage
  6. OPTIONAL: Implement export_to_csv() for CSV output
================================================================================
"""

import csv
import json
import os
import sys
from typing import Dict, Any

DATABASE_FILE = "sample_records.json"

# Temporary registry: Student ID is the key and student details are stored as a dictionary
STUDENT_REGISTRY: Dict[str, Dict[str, Any]] = {}


def load_records_from_json(file_path: str) -> Dict[str, Dict[str, Any]]:
    """Reads student information from a JSON file without stopping the program on common file errors."""

    if not os.path.exists(file_path):
        print(f"[WARN] Database file '{file_path}' not found. Starting with empty registry.")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            print(f"[SUCCESS] Loaded {len(data)} record(s) from {file_path}.")
            return data

    except json.JSONDecodeError as json_err:
        print(f"[ERROR] Corrupted JSON structure in '{file_path}': {json_err}")
        return {}

    except Exception as err:
        print(f"[UNEXPECTED ERROR] Failed to load data: {err}")
        return {}


def view_all_records(registry: Dict[str, Dict[str, Any]]) -> None:
    """Displays the stored student information in a clear table."""

    if not registry:
        print("\n[INFO] No records found in the registry.")
        return

    separator = "-" * 75

    print("\n" + separator)
    print(f"{'Student ID':<12} | {'Name':<22} | {'Branch':<22} | {'CGPA':<5}")
    print(separator)

    for student_id, details in registry.items():
        name = details.get("name", "N/A")
        branch = details.get("branch", "N/A")
        cgpa = details.get("cgpa", 0.0)

        print(
            f"{student_id:<12} | "
            f"{name:<22} | "
            f"{branch:<22} | "
            f"{cgpa:<5.2f}"
        )

    print(separator + "\n")


# ==============================================================================
# ✍️ IMPLEMENTATION SECTION
# ==============================================================================


def add_student_record(registry: Dict[str, Dict[str, Any]]) -> None:

    print("\n--- Register Student ---")

    student_id = input("Student ID: ").strip()

    if not student_id:
        print("[ERROR] Student ID is required.")
        return

    if student_id in registry:
        print("[ERROR] This Student ID is already registered.")
        return

    name = input("Student Name: ").strip()

    if not name:
        print("[ERROR] Student name is required.")
        return

    branch = input("Program/Branch: ").strip()

    if not branch:
        print("[ERROR] Program/Branch is required.")
        return

    cgpa_input = input("CGPA (0.0 - 10.0): ").strip()

    try:
        cgpa = float(cgpa_input)

        if cgpa < 0.0 or cgpa > 10.0:
            print("[ERROR] CGPA should be within 0.0 and 10.0.")
            return

    except ValueError:
        print("[ERROR] Please enter CGPA as a numeric value.")
        return

    first_name = name.split()[0].lower()
    email = f"{first_name}.{student_id.lower()}@university.edu"

    registry[student_id] = {
        "name": name,
        "branch": branch,
        "cgpa": cgpa,
        "email": email
    }

    print(f"[SUCCESS] Student '{name}' has been registered successfully.")
    print(f"[INFO] Email created: {email}")


def search_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Search for a student using ID or a partial name."""

    print("\n--- Find Student Records ---")

    if not registry:
        print("[INFO] There are no student entries to search.")
        return

    search_term = input("Enter ID or student name: ").strip()

    if not search_term:
        print("[ERROR] Search input cannot be blank.")
        return

    # Exact ID search
    if search_term in registry:

        details = registry[search_term]

        print("\n[SUCCESS] Record Found")
        print("-" * 40)
        print(f"Student ID : {search_term}")
        print(f"Name       : {details.get('name', 'N/A')}")
        print(f"Branch     : {details.get('branch', 'N/A')}")
        print(f"CGPA       : {details.get('cgpa', 0.0):.2f}")
        print(f"Email      : {details.get('email', 'N/A')}")
        print("-" * 40)

        return

    # Partial and case-insensitive name search
    normalized_query = search_term.lower()
    found = False

    for student_id, details in registry.items():

        name = details.get("name", "")

        if normalized_query in name.lower():

            if not found:
                print("\nMatching Student Records:")
                print("-" * 75)

                print(
                    f"{'Student ID':<12} | "
                    f"{'Name':<22} | "
                    f"{'Branch':<22} | "
                    f"{'CGPA':<5}"
                )

                print("-" * 75)

            print(
                f"{student_id:<12} | "
                f"{name:<22} | "
                f"{details.get('branch', 'N/A'):<22} | "
                f"{details.get('cgpa', 0.0):<5.2f}"
            )

            found = True

    if found:
        print("-" * 75)
    else:
        print("[INFO] No student record matched the search.")


def delete_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Remove a student after confirmation."""

    print("\n--- Remove Student Record ---")

    if not registry:
        print("[INFO] There are no records available for removal.")
        return

    student_id = input("Student ID to remove: ").strip()

    if student_id not in registry:
        print("[ERROR] No record exists for this Student ID.")
        return

    student = registry[student_id]

    print("\nSelected record:")
    print(f"Name   : {student.get('name', 'N/A')}")
    print(f"Branch : {student.get('branch', 'N/A')}")
    print(f"CGPA   : {student.get('cgpa', 0.0):.2f}")

    confirmation = input("Confirm removal? (y/n): ").strip().lower()

    if confirmation in ("y", "yes"):
        del registry[student_id]
        print("[SUCCESS] Student record removed successfully.")
    else:
        print("[INFO] Removal cancelled.")


def save_records_to_json(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    """Store the current registry dictionary in JSON format with error handling."""

    try:

        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(registry, file, indent=2)

        print(f"[SUCCESS] Saved {len(registry)} record(s) to {file_path}.")

    except Exception as err:
        print(f"[ERROR] Failed to save records: {err}")


def export_to_csv(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    """Write all registered students into a CSV file."""

    if not registry:
        print("[INFO] There are no records available for export.")
        return

    try:

        fieldnames = [
            "student_id",
            "name",
            "branch",
            "cgpa",
            "email"
        ]

        with open(
            file_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as file:

            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames
            )

            writer.writeheader()

            for student_id, details in registry.items():

                writer.writerow({
                    "student_id": student_id,
                    "name": details.get("name", ""),
                    "branch": details.get("branch", ""),
                    "cgpa": details.get("cgpa", 0.0),
                    "email": details.get("email", "")
                })

        print(
            f"[SUCCESS] Exported {len(registry)} "
            f"record(s) into {file_path}."
        )

    except Exception as err:
        print(f"[ERROR] Failed to export records: {err}")


def main_menu() -> None:
    """Controls the main command-line menu."""

    global STUDENT_REGISTRY

    STUDENT_REGISTRY = load_records_from_json(DATABASE_FILE)

    menu_banner = """
========================================
🎓 STUDENT INFORMATION REGISTRY
========================================
1. Display All Students
2. Register Student
3. Find Student
4. Remove Student
5. Save Data
6. Export to CSV (Optional)
0. Save & Close
========================================
"""

    while True:

        print(menu_banner)

        choice = input("Select option [0-6]: ").strip()

        if choice == "1":
            view_all_records(STUDENT_REGISTRY)

        elif choice == "2":
            add_student_record(STUDENT_REGISTRY)

        elif choice == "3":
            search_student_record(STUDENT_REGISTRY)

        elif choice == "4":
            delete_student_record(STUDENT_REGISTRY)

        elif choice == "5":
            save_records_to_json(
                DATABASE_FILE,
                STUDENT_REGISTRY
            )

        elif choice == "6":
            export_to_csv(
                "students_export.csv",
                STUDENT_REGISTRY
            )

        elif choice == "0":

            save_records_to_json(
                DATABASE_FILE,
                STUDENT_REGISTRY
            )

            print("[INFO] Registry closed successfully. Goodbye!")
            sys.exit(0)

        else:
            print(
                "[WARN] Invalid selection. "
                "Choose a number from 0 to 6."
            )


if __name__ == "__main__":
    main_menu()

[WARN] Database file 'sample_records.json' not found. Starting with empty registry.

🎓 STUDENT INFORMATION REGISTRY
1. Display All Students
2. Register Student
3. Find Student
4. Remove Student
5. Save Data
6. Export to CSV (Optional)
0. Save & Close

Select option [0-6]: 2

--- Register Student ---
Student ID: 2345678
Student Name: vedika
Program/Branch: btech
CGPA (0.0 - 10.0): 8
[SUCCESS] Student 'vedika' has been registered successfully.
[INFO] Email created: vedika.2345678@university.edu

🎓 STUDENT INFORMATION REGISTRY
1. Display All Students
2. Register Student
3. Find Student
4. Remove Student
5. Save Data
6. Export to CSV (Optional)
0. Save & Close

Select option [0-6]: 0
[SUCCESS] Saved 1 record(s) to sample_records.json.
[INFO] Registry closed successfully. Goodbye!


SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
